# 04 — Retrieval-Enhanced Pipeline

This notebook walks through:

1. Loading the sentence-transformer embeddings of all 14,961 MDCC campaigns.
2. Building / loading the FAISS index for fast cosine-similarity search.
3. Querying the index with a few example texts to see what 'neighbour
   evidence' looks like in practice.
4. Combining the transformer prediction with a neighbour soft-vote to get
   the retrieval-enhanced prediction.

**Educational note.** A sentence-transformer maps a sentence into a 384-dim
vector where semantically similar sentences are close together. FAISS is a
library that does fast nearest-neighbour search in that space — it scales to
billions of vectors. We use the simplest variant, `IndexFlatIP`, because at
~15k vectors brute-force is fast enough and exact.


In [1]:
# Path-setup boilerplate so the notebook can import src.*
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("project root:", ROOT)


project root: /Users/spandarayamajhi/Desktop/Artificial Intelligence Coursework (Final Submission)


In [2]:
import numpy as np, pandas as pd
from src import config
from src import retrieval_pipeline as rp
from src.models import retrieval_enhanced_classifier as rec


## 1. Load saved embeddings + index

In [3]:
embs = np.load(config.EMBEDDINGS_NPY)
index = rp.load_index(config.FAISS_INDEX)
print('embeddings shape:', embs.shape)
print('FAISS ntotal :', index.ntotal)


embeddings shape: (14961, 384)
FAISS ntotal : 14961


## 2. Try retrieval on three queries

In [4]:
df = pd.read_csv(config.PROCESSED_CSV)
queries = [
    'Please help — only days left, my brother will lose his home',
    'We are raising funds to cover medical bills after a surgery',
    'It breaks our heart to ask but every second counts to save her life',
]

# Embed the queries with the same model used for the index
q_emb = rp.embed(queries, show_progress=False)
sims, idx = rp.topk_neighbours(index, q_emb, k=3)

for q, ii, ss in zip(queries, idx, sims):
    print('QUERY :', q[:80])
    for j, s in zip(ii, ss):
        print(f'  sim={s:.3f}  cat={df.iloc[j].category}  '
              f'label={df.iloc[j].binary_label}  '
              f'text="{df.iloc[j].text[:80]}..."')
    print()


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

QUERY : Please help — only days left, my brother will lose his home
  sim=0.613  cat=Emergency  label=non_manipulative  text="Hi, Paul and I are starting this gofund me to help his sister and her family who..."
  sim=0.603  cat=Financial Emergency  label=non_manipulative  text="Hello my name is Kymberly and I'm Stuartt's big sister. On November 10th, he los..."
  sim=0.589  cat=Emergency  label=non_manipulative  text="Hey everyone, Tuesday morning I found out I lost my Little Brother Tony, I know ..."

QUERY : We are raising funds to cover medical bills after a surgery
  sim=0.727  cat=Animals  label=non_manipulative  text="Hello, my name is Daxton and I’m currently raising funds to help pay for my dogs..."
  sim=0.716  cat=Medical  label=non_manipulative  text="I am raising funds to cover my bills and living expenses while recovering from o..."
  sim=0.698  cat=Financial Emergency  label=non_manipulative  text="Hi, I'm Robert and I created this go fund me to raise money keep my bills 

Inspect the output above. You should see that queries with strong urgency
language retrieve neighbours from Memorial / Emergency categories with
*manipulative* labels, while the neutral medical-bills query retrieves
*non_manipulative* neighbours.


## 3. Retrieval-enhanced classification

In [5]:
# Reuse the transformer's saved test-set predictions
bert_preds = np.load(config.RESULTS_DIR / 'distilbert_preds.npz',
                     allow_pickle=True)
test_texts = bert_preds['X_test'].tolist()
y_true = bert_preds['y_true']
bert_proba = bert_preds['y_proba']

# Map texts -> df indices to recover train-side embeddings
text2i = {t: i for i, t in enumerate(df['text'].astype(str).tolist())}
test_idx = np.array([text2i[t] for t in test_texts])
mask = np.ones(len(df), bool); mask[test_idx] = False
train_idx = np.where(mask)[0]

train_emb = embs[train_idx]
test_emb  = embs[test_idx]
train_lab = df['binary_label'].map(config.LABEL2ID).values[train_idx]
train_index = rp.build_index(train_emb)
sims, neigh = rp.topk_neighbours(train_index, test_emb, k=config.RETRIEVAL_TOP_K)

ypred, yproba = rec.predict_combined(bert_proba, neigh, sims, train_lab, alpha=0.7)
from src.evaluation import compute_metrics
m = compute_metrics(y_true, ypred, yproba, 'Retrieval-Enhanced',
                    labels=config.LABELS_BINARY)
print(f'accuracy {m.accuracy:.4f}  f1 {m.f1:.4f}  macro_f1 {m.macro_f1:.4f}')


accuracy 0.7113  f1 0.6362  macro_f1 0.6984


Expected: ~0.7113 accuracy / 0.6362 F1 — *lower* than the standalone
transformer at α=1.0. The α-ablation in notebook 05 makes this explicit.
